In [ ]:
## 서울, 경기, 강원, 부산, 인천, 충남, 대전, 충북, 세종, 경북, 대구

In [10]:
import time
import pandas as pd
import os
import certifi
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# SSL 인증서 경로 설정 (지난번 에러 방지)
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

def crawl_ediya_seoul():
    # 1. 서울시 구 리스트
    seoul_gu_list = [
        "강남구", "강동구", "강북구", "강서구", "관악구", "광진구", "구로구", "금천구", 
        "노원구", "도봉구", "동대문구", "동작구", "마포구", "서대문구", "서초구", "성동구", 
        "성북구", "송파구", "양천구", "영등포구", "용산구", "은평구", "종로구", "중구", "중랑구"
    ]

    options = webdriver.ChromeOptions()
    # options.add_argument('--headless') # 창을 숨기고 싶으면 주석 해제
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://members.ediya.com/store#none"
    driver.get(url)
    
    all_stores = []

    try:
        # [중요] 2. '주소' 탭 클릭하기
        # 주소 검색 탭을 명시적으로 클릭하여 활성화합니다.
        address_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@onclick, 'storeAddr')]")))
        address_tab.click()
        time.sleep(1) # 탭 전환 애니메이션 대기

        for gu in seoul_gu_list:
            start_time = time.time() # 8. 구별 시작 시간 측정
            
            try:
                # 3. 주소 입력칸에 구 이름 입력
                search_input = wait.until(EC.presence_of_element_located((By.ID, "keyword")))
                search_input.clear()
                search_input.send_keys(f"{gu}")
                
                # 4. 검색 버튼 클릭 (JavaScript 실행 방식으로 더 확실하게 클릭)
                search_btn = driver.find_element(By.CLASS_NAME, "btn_search")
                driver.execute_script("arguments[0].click();", search_btn)
                
                # 5. 결과 로딩 대기
                time.sleep(2) 
                
                # 6. BeautifulSoup 파싱
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                
                # 'store_list' 내부의 'st_li' 클래스들을 모두 가져옴
                items = soup.select('.store_list .st_li')
                
                gu_store_count = 0
                for item in items:
                    name = item.select_one('.name').get_text(strip=True)
                    addr = item.select_one('.addr').get_text(strip=True)
                    
                    all_stores.append({
                        '매장명': name,
                        '주소': addr,
                        '구': gu
                    })
                    gu_store_count += 1
                
                # 8. 구별 소요 시간 콘솔 출력
                end_time = time.time()
                print(f"[{gu}] 완료! - 매장 수: {gu_store_count:2d}개 | 소요시간: {end_time - start_time:.2f}초")

            except Exception as e:
                print(f"{gu} 크롤링 중 에러 발생: {e}")

    finally:
        # 10. CSV 파일 저장 (데이터가 있을 경우에만)
        if all_stores:
            df = pd.DataFrame(all_stores)
            # 중복 데이터 제거 (검색 결과가 겹칠 수 있음)
            df = df.drop_duplicates(subset=['매장명', '주소'])
            df.to_csv("ediya.csv", index=False, encoding="utf-8-sig")
            print(f"\n총 {len(df)}개의 유니크한 매장 정보를 CSV로 저장했습니다.")
        
        driver.quit()

if __name__ == "__main__":
    crawl_ediya_seoul()

[강남구] 완료! - 매장 수: 62개 | 소요시간: 2.93초
[강동구] 완료! - 매장 수: 38개 | 소요시간: 3.17초
[강북구] 완료! - 매장 수: 18개 | 소요시간: 2.79초
[강서구] 완료! - 매장 수: 50개 | 소요시간: 3.03초
[관악구] 완료! - 매장 수: 32개 | 소요시간: 2.58초
[광진구] 완료! - 매장 수: 30개 | 소요시간: 2.54초
[구로구] 완료! - 매장 수: 32개 | 소요시간: 2.57초
[금천구] 완료! - 매장 수: 28개 | 소요시간: 2.52초
[노원구] 완료! - 매장 수: 34개 | 소요시간: 2.52초
[도봉구] 완료! - 매장 수: 28개 | 소요시간: 2.56초
[동대문구] 완료! - 매장 수: 36개 | 소요시간: 2.58초
[동작구] 완료! - 매장 수: 36개 | 소요시간: 2.57초
[마포구] 완료! - 매장 수: 34개 | 소요시간: 2.57초
[서대문구] 완료! - 매장 수: 30개 | 소요시간: 2.52초
[서초구] 완료! - 매장 수: 52개 | 소요시간: 2.60초
[성동구] 완료! - 매장 수: 28개 | 소요시간: 2.54초
[성북구] 완료! - 매장 수: 38개 | 소요시간: 2.56초
[송파구] 완료! - 매장 수: 38개 | 소요시간: 2.61초
[양천구] 완료! - 매장 수: 20개 | 소요시간: 2.55초
[영등포구] 완료! - 매장 수: 68개 | 소요시간: 4.84초
[용산구] 완료! - 매장 수: 30개 | 소요시간: 2.53초
[은평구] 완료! - 매장 수: 40개 | 소요시간: 2.56초
[종로구] 완료! - 매장 수: 56개 | 소요시간: 2.72초
[중구] 완료! - 매장 수: 100개 | 소요시간: 2.86초
[중랑구] 완료! - 매장 수: 48개 | 소요시간: 2.71초

총 503개의 유니크한 매장 정보를 CSV로 저장했습니다.


In [5]:
import time
import pandas as pd
import os
import certifi
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# SSL 인증서 에러 방지
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

def crawl_ediya_gyeongbuk_append():
# 1. 경상북도 상세 리스트 (포항 2개구 포함, 군위 제외)
    gyeongbuk_list = [
        "포항시", "경주시", "김천시", "안동시", "구미시", 
        "영주시", "상주시", "문경시", "경산시", "영천시", "의성군", "청송군", 
        "영양군", "영덕군", "청도군", "고령군", "성주군", "칠곡군", "예천군", 
        "봉화군", "울진군", "울릉군"
    ]

    target_file = "ediya.csv"
    
    # [중복 방지용] 기존 파일 로드 및 기존 지점 세트 생성
    seen_stores = set()
    if os.path.exists(target_file):
        existing_df = pd.read_csv(target_file)
        # '매장명'과 '주소'를 합쳐서 유니크한 키 생성
        for _, row in existing_df.iterrows():
            seen_stores.add((row['매장명'], row['주소']))
        print(f"기존 파일에서 {len(seen_stores)}개의 지점을 확인했습니다. 중복을 제외하고 수집합니다.")

    options = webdriver.ChromeOptions()
    # options.add_argument('--headless')
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://members.ediya.com/store#none"
    driver.get(url)
    
    new_stores = []

    try:
        # 2. '주소' 탭 클릭
        address_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@onclick, 'storeAddr')]")))
        address_tab.click()
        time.sleep(1)

        for area in gyeongbuk_list:
            start_time = time.time()
            current_area_new_count = 0
            
            try:
                # 3. 주소 입력칸에 '경상북도 시/군' 입력
                search_input = wait.until(EC.presence_of_element_located((By.ID, "keyword")))
                search_input.clear()
                search_input.send_keys(f"{area}")
                
                # 4. 검색 버튼 클릭
                search_btn = driver.find_element(By.CLASS_NAME, "btn_search")
                driver.execute_script("arguments[0].click();", search_btn)
                
                # 5. 결과 로딩 대기 (검색 결과가 바뀔 시간을 충분히 줌)
                time.sleep(2.5) 
                
                # 6. BeautifulSoup 파싱
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                items = soup.select('.store_list .st_li')
                
                for item in items:
                    name = item.select_one('.name').get_text(strip=True)
                    addr = item.select_one('.addr').get_text(strip=True)
                    
                    # [중복 체크] 이름과 주소가 모두 같으면 수집하지 않음
                    if (name, addr) not in seen_stores:
                        new_stores.append({
                            '매장명': name,
                            '주소': addr,
                            '지역': area
                        })
                        seen_stores.add((name, addr)) # 수집 목록에 추가
                        current_area_new_count += 1
                
                # 8. 소요 시간 출력
                end_time = time.time()
                print(f"[{area}] 완료! - 새로 추가된 매장: {current_area_new_count:2d}개 | 소요시간: {end_time - start_time:.2f}초")

            except Exception as e:
                print(f"{area} 크롤링 중 에러 발생: {e}")

    finally:
        # 10. CSV 파일에 추가 저장
        if new_stores:
            new_df = pd.DataFrame(new_stores)
            
            # 파일이 있으면 헤더 없이 추가(append), 없으면 새로 생성
            if os.path.exists(target_file):
                new_df.to_csv(target_file, index=False, encoding="utf-8-sig", mode='a', header=False)
                print(f"\n--- {target_file}에 경상북도 데이터 {len(new_stores)}건을 추가했습니다. ---")
            else:
                new_df.to_csv(target_file, index=False, encoding="utf-8-sig")
                print(f"\n--- {target_file}이 없어 새로 생성하고 저장했습니다. ---")
        else:
            print("\n새로 수집된 중복되지 않은 데이터가 없습니다.")
        
        driver.quit()

if __name__ == "__main__":
    crawl_ediya_gyeongbuk_append()

기존 파일에서 1702개의 지점을 확인했습니다. 중복을 제외하고 수집합니다.
[포항시] 완료! - 새로 추가된 매장: 24개 | 소요시간: 3.42초
[경주시] 완료! - 새로 추가된 매장: 37개 | 소요시간: 3.52초
[김천시] 완료! - 새로 추가된 매장:  6개 | 소요시간: 3.19초
[안동시] 완료! - 새로 추가된 매장:  8개 | 소요시간: 3.13초
[구미시] 완료! - 새로 추가된 매장: 17개 | 소요시간: 3.05초
[영주시] 완료! - 새로 추가된 매장:  5개 | 소요시간: 2.95초
[상주시] 완료! - 새로 추가된 매장:  4개 | 소요시간: 2.89초
[문경시] 완료! - 새로 추가된 매장:  4개 | 소요시간: 2.94초
[경산시] 완료! - 새로 추가된 매장:  9개 | 소요시간: 3.04초
[영천시] 완료! - 새로 추가된 매장:  3개 | 소요시간: 2.93초
[의성군] 완료! - 새로 추가된 매장:  2개 | 소요시간: 2.90초
[청송군] 완료! - 새로 추가된 매장:  1개 | 소요시간: 2.92초
[영양군] 완료! - 새로 추가된 매장:  1개 | 소요시간: 3.56초
[영덕군] 완료! - 새로 추가된 매장:  5개 | 소요시간: 2.92초
[청도군] 완료! - 새로 추가된 매장:  2개 | 소요시간: 2.88초
[고령군] 완료! - 새로 추가된 매장:  0개 | 소요시간: 2.78초
[성주군] 완료! - 새로 추가된 매장:  0개 | 소요시간: 2.75초
[칠곡군] 완료! - 새로 추가된 매장:  5개 | 소요시간: 2.94초
[예천군] 완료! - 새로 추가된 매장:  3개 | 소요시간: 2.93초
[봉화군] 완료! - 새로 추가된 매장:  3개 | 소요시간: 2.90초
[울진군] 완료! - 새로 추가된 매장:  5개 | 소요시간: 2.97초
[울릉군] 완료! - 새로 추가된 매장:  0개 | 소요시간: 2.77초

--- ediya.csv에 경상북도 데이터 144건을 추가했습니다. ---


In [6]:
import time
import pandas as pd
import os
import certifi
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# SSL 인증서 에러 방지
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

def crawl_ediya_daegu_append():
    # 1. 대구광역시 구/군 리스트 (군위군 포함)
    daegu_list = ["중구", "동구", "서구", "남구", "북구", "수성구", "달서구", "달성군", "군위군"]

    target_file = "ediya.csv"
    
    # [중복 방지] 기존 파일 로드 (서울, 경기 데이터가 이미 있다면 그 지점들 기억)
    seen_stores = set()
    if os.path.exists(target_file):
        existing_df = pd.read_csv(target_file)
        for _, row in existing_df.iterrows():
            seen_stores.add((row['매장명'], row['주소']))
        print(f"현재 파일에 저장된 {len(seen_stores)}개 지점을 건너뛰고 수집합니다.")

    options = webdriver.ChromeOptions()
    # options.add_argument('--headless')
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 10)
    
    url = "https://members.ediya.com/store#none"
    driver.get(url)
    
    new_stores = []

    try:
        # 2. '주소' 탭 클릭
        address_tab = wait.until(EC.element_to_be_clickable((By.XPATH, "//a[contains(@onclick, 'storeAddr')]")))
        address_tab.click()
        time.sleep(1)

        for area in daegu_list:
            start_time = time.time()
            current_area_new_count = 0
            
            try:
                # 3. 주소 입력칸에 '{area}' 입력
                search_input = wait.until(EC.presence_of_element_located((By.ID, "keyword")))
                search_input.clear()
                search_input.send_keys(f"{area}")
                
                # 4. 검색 버튼 클릭
                search_btn = driver.find_element(By.CLASS_NAME, "btn_search")
                driver.execute_script("arguments[0].click();", search_btn)
                
                # 5. 결과 로딩 대기
                time.sleep(2.5) 
                
                # 6. BeautifulSoup 파싱
                html = driver.page_source
                soup = BeautifulSoup(html, 'html.parser')
                items = soup.select('.store_list .st_li')
                
                for item in items:
                    name_tag = item.select_one('.name')
                    addr_tag = item.select_one('.addr')
                    
                    if name_tag and addr_tag:
                        name = name_tag.get_text(strip=True)
                        addr = addr_tag.get_text(strip=True)
                        
                        # [중복 체크] 이미 수집된 지점인지 확인
                        if (name, addr) not in seen_stores:
                            new_stores.append({
                                '매장명': name,
                                '주소': addr,
                                '지역': area
                            })
                            seen_stores.add((name, addr))
                            current_area_new_count += 1
                
                end_time = time.time()
                print(f"[{area}] 완료 - 신규 추가: {current_area_new_count:2d}개 | 소요시간: {end_time - start_time:.2f}초")

            except Exception as e:
                print(f"{area} 크롤링 중 오류: {e}")

    finally:
        # 10. CSV 파일에 추가 저장
        if new_stores:
            new_df = pd.DataFrame(new_stores)
            if os.path.exists(target_file):
                new_df.to_csv(target_file, index=False, encoding="utf-8-sig", mode='a', header=False)
                print(f"\n--- {target_file}에 대구 데이터 {len(new_stores)}건을 추가했습니다. ---")
            else:
                new_df.to_csv(target_file, index=False, encoding="utf-8-sig")
                print(f"\n--- {target_file}을 새로 생성하고 대구 데이터를 저장했습니다. ---")
        else:
            print("\n추가할 신규 데이터가 없습니다.")
        
        driver.quit()

if __name__ == "__main__":
    crawl_ediya_daegu_append()

현재 파일에 저장된 1846개 지점을 건너뛰고 수집합니다.
[중구] 완료 - 신규 추가:  0개 | 소요시간: 3.70초
[동구] 완료 - 신규 추가:  0개 | 소요시간: 3.53초
[서구] 완료 - 신규 추가:  0개 | 소요시간: 3.85초
[남구] 완료 - 신규 추가:  0개 | 소요시간: 4.08초
[북구] 완료 - 신규 추가:  0개 | 소요시간: 3.79초
[수성구] 완료 - 신규 추가: 20개 | 소요시간: 3.22초
[달서구] 완료 - 신규 추가: 15개 | 소요시간: 3.19초
[달성군] 완료 - 신규 추가: 14개 | 소요시간: 3.05초
[군위군] 완료 - 신규 추가:  1개 | 소요시간: 2.92초

--- ediya.csv에 대구 데이터 50건을 추가했습니다. ---
